# Lab 2.3 &mdash; Sub-goals That Finish, and Knowing When to Re-plan

**Level:** Advanced &nbsp;|&nbsp; **Est. time:** 35 min &nbsp;|&nbsp; **Day 1 &middot; Module 2 &mdash; Agentic Planning &amp; Reasoning**

### What you'll do
- Write the test that separates a finishable sub-goal from a wish
- Classify a failure as transient or a wrong plan -- from the error, not a guess
- Implement the four-rung escalation ladder with a bound on every rung
- Watch a retry-only agent and a diagnosing agent meet the same failures

> **How this lab works.** Fill every `BLANK`, then run the **Self-check** cell under each section.
> Graded cells are plain Python and never call a model, so your score never depends on a
> live endpoint. Cells marked **Run it for real** do call the sandbox model; if it is not
> reachable they print how to fix it instead of crashing.

> **Builds on Lab 1.2's `order_steps`.** There you ordered a plan someone gave you.
> Here you judge whether the steps were worth ordering, and what to do when one fails.

In [ ]:
# ---------------------------------------------------------------- Setup: run me first
import os, json, time, textwrap
from typing import Any, Callable

WORK = os.path.join("/tmp", "awmas-lab-2-03")
os.makedirs(WORK, exist_ok=True)

# ---- self-check plumbing -------------------------------------------------
_results = []

def check(name: str, fn: Callable[[], Any], hint: str = "") -> None:
    """[PASS] / [FAIL] / [TODO] for one assertion. An unfilled blank prints [TODO]."""
    try:
        ok = bool(fn())
    except NameError:
        print(f"[TODO] {name}")
        _results.append(None)
        return
    except Exception as exc:
        print(f"[FAIL] {name} -- {type(exc).__name__}: {exc}")
        _results.append(False)
        return
    print(("[PASS] " if ok else "[FAIL] ") + name + ("" if ok else (" -- " + hint if hint else "")))
    _results.append(ok)

def guard(fn: Callable[[], Any], default: Any = None) -> Any:
    """Run fn(). If a blank above is still unfilled, say so and carry on -- never crash Run All."""
    try:
        return fn()
    except NameError:
        print("(a blank above is still unfilled -- fill it in, then re-run this cell)")
        return default

def score() -> None:
    done = [r for r in _results if r is not None]
    passed = sum(1 for r in done if r)
    todo = sum(1 for r in _results if r is None)
    print(f"\nScore: {passed}/{len(_results)}" + (f"   ({todo} still TODO)" if todo else ""))

# ---- the sandbox model ---------------------------------------------------
# Your sandbox already has an LLM configured -- nothing to install, no key to register.
# These two values are read from the environment so this notebook never hardcodes an endpoint.
LLM_BASE_URL = (os.environ.get("LAB_LLM_BASE_URL") or os.environ.get("OPENAI_BASE_URL")
                or os.environ.get("LITELLM_BASE_URL"))
LLM_MODEL    = (os.environ.get("LAB_LLM_MODEL") or os.environ.get("OPENAI_MODEL")
                or os.environ.get("LITELLM_MODEL"))
LLM_API_KEY  = os.environ.get("OPENAI_API_KEY", "sandbox")

def llm_ready() -> bool:
    if not LLM_BASE_URL or not LLM_MODEL:
        print("Model not configured. In a sandbox terminal run `env | grep -i llm` and set:")
        print("  export LAB_LLM_BASE_URL=...    # the gateway URL from your welcome sheet")
        print("  export LAB_LLM_MODEL=...       # the model name from your welcome sheet")
        return False
    return True

_llm = None
def get_llm(temperature: float = 0.0):
    """A LangChain chat model pointed at the sandbox gateway (OpenAI-compatible)."""
    global _llm
    if _llm is None:
        from langchain_openai import ChatOpenAI
        _llm = ChatOpenAI(model=LLM_MODEL, base_url=LLM_BASE_URL,
                          api_key=LLM_API_KEY, temperature=temperature)
    return _llm

def ask(prompt: str, system: str | None = None) -> str:
    """One stateless call. Returns text, or an error string -- never raises."""
    try:
        msgs = ([("system", system)] if system else []) + [("human", prompt)]
        return get_llm().invoke(msgs).content
    except Exception as exc:
        return f"<model unavailable: {type(exc).__name__}: {exc}>"

print("work dir:", WORK)
print("model   :", LLM_MODEL or "(not configured -- graded cells still work)")

In [ ]:
# ------------------------------------------------- the case file (synthetic, self-contained)
# One domain runs through all five Module 1 labs: payment exceptions on a small ledger.
# Nothing here is real data and nothing leaves this notebook.

LEDGER = {
    "PMT-1001": {"amount": 250000.00, "ccy": "USD", "counterparty": "NORTHWIND",
                 "status": "settled",  "value_date": "2026-09-01", "reason_code": None},
    "PMT-1002": {"amount":  48250.75, "ccy": "EUR", "counterparty": "ACME-EU",
                 "status": "failed",   "value_date": "2026-09-02", "reason_code": "INSUFFICIENT_FUNDS"},
    "PMT-1003": {"amount": 990000.00, "ccy": "USD", "counterparty": "ZENITH",
                 "status": "held",     "value_date": "2026-09-02", "reason_code": "LIMIT_BREACH"},
    "PMT-1004": {"amount":   1200.00, "ccy": "GBP", "counterparty": "ACME-UK",
                 "status": "failed",   "value_date": "2026-09-03", "reason_code": "INVALID_IBAN"},
    "PMT-1005": {"amount": 750000.00, "ccy": "USD", "counterparty": "NORTHWIND",
                 "status": "held",     "value_date": "2026-09-03", "reason_code": "SANCTIONS_REVIEW"},
}

POLICY = {
    "INSUFFICIENT_FUNDS": "Retry once after 24h. If it fails again, notify the client desk. No manual funding.",
    "LIMIT_BREACH":       "Payments above USD 500,000 need Treasury approval before release.",
    "INVALID_IBAN":       "Return to originator with code R04. Never repair beneficiary details in-house.",
    "SANCTIONS_REVIEW":   "Hold. Compliance decides. Operations must not release or cancel.",
}

# Which reason codes may an agent resolve on its own, and which need a human?
NEEDS_HUMAN = {"LIMIT_BREACH", "SANCTIONS_REVIEW"}

print(f"{len(LEDGER)} payments, {len(POLICY)} policy rules loaded")

In [ ]:
# ------------------------------------------------- carried forward from Lab 1.2 of Module 1
# These are the tools you wrote in Lab 1.2 of Module 1. Nothing to fill in -- they are here so this
# notebook runs on its own. Note the docstrings: they name the case AND the boundary.

def lookup_payment(ref: str) -> str:
    """Return the ledger record for one payment reference such as 'PMT-1002'.

    Use when you need the status, amount, counterparty or reason code of a specific payment.
    Not for searching across payments.
    """
    record = LEDGER.get(ref)
    if record is None:
        return f"no payment found with reference {ref!r}"
    return json.dumps({"ref": ref, **record})


def policy_for(reason_code: str) -> str:
    """Return the operating policy for one failure reason code, e.g. 'LIMIT_BREACH'.

    Use after you know why a payment failed and need to know what to do about it.
    """
    return POLICY.get(reason_code, f"no policy on file for reason code {reason_code!r}")


TOOLS = {"lookup_payment": lookup_payment, "policy_for": policy_for}
print("carried forward:", ", ".join(TOOLS))

## Concept

Two failures look identical from inside the loop and need opposite responses:

- **Transient** &mdash; the plan was fine, the world was briefly busy. **Retry**, bounded.
- **Wrong plan** &mdash; the step cannot succeed as written. **Re-plan**, feeding the error back.

Retry a wrong plan and you buy the same failure repeatedly. Re-plan a blip and you throw away
correct work. The diagnosis is the whole job, and the error your tools return is the evidence.

## Section 1 &mdash; Is this sub-goal finishable?

A sub-goal an agent cannot finish becomes a loop that gets misdiagnosed as a reasoning bug. The
test is mechanical: is there an output that would settle it?

In [ ]:
VAGUE = ("understand", "thoroughly", "make sure", "as needed", "properly",
         "fully", "appropriate", "confident", "comprehensive")

CONCRETE = ("return", "fetch", "compute", "classify", "state", "list")

def is_finishable(subgoal: str) -> bool:
    """True when a sub-goal has a detectable definition of done.

    Rejects a sub-goal that contains a vague qualifier from VAGUE, or that does not
    start with a concrete verb naming what will be produced.
    """
    text = subgoal.lower().strip()
    if any(w in text for w in VAGUE):
        return False
    return text.startswith(CONCRETE)

In [ ]:
# --- Self-check: Section 1
check("a concrete sub-goal passes",
      lambda: is_finishable("Return the reason code for PMT-1002") is True)
check("a vague qualifier fails it",
      lambda: is_finishable("Understand the payment problem") is False)
check("'make sure nothing is missed' fails",
      lambda: is_finishable("Make sure nothing has been missed") is False)
check("a concrete verb is required",
      lambda: is_finishable("Look into the ledger a bit") is False,
      "if the step does not name an output, its completion is not detectable")
check("'state whether ...' passes",
      lambda: is_finishable("State whether policy reserves this for a human") is True)
check("'investigate until confident' fails",
      lambda: is_finishable("Investigate until confident") is False)

## Section 2 &mdash; Diagnose the failure

The error a tool returns is data. Classify on it rather than on intuition, and the right response
follows automatically.

In [ ]:
TRANSIENT = ("timeout", "timed out", "503", "502", "429", "connection reset",
             "temporarily unavailable", "deadlock", "try again")
PERMANENT = ("404", "not found", "no payment found", "403", "forbidden",
             "invalid", "no such", "malformed", "unauthorised")

def diagnose(error: str) -> str:
    """Return 'transient', 'wrong_plan' or 'unknown' for one error string."""
    e = error.lower()
    if any(t in e for t in TRANSIENT):
        return "transient"
    if any(p in e for p in PERMANENT):
        return "wrong_plan"
    return "unknown"

def respond_to(diagnosis: str, attempts: int, max_attempts: int = 3) -> str:
    """The rung of the ladder to take. One of: retry, replan, escalate."""
    if diagnosis == "transient":
        return "retry" if attempts < max_attempts else "escalate"
    if diagnosis == "wrong_plan":
        return "replan"
    return "escalate"                # unknown errors go to a human, never to a guess

In [ ]:
# --- Self-check: Section 2
check("a 503 is transient", lambda: diagnose("HTTP 503 service unavailable") == "transient")
check("a timeout is transient", lambda: diagnose("read timed out after 30s") == "transient")
check("a 404 is a wrong plan", lambda: diagnose("HTTP 404 not found") == "wrong_plan")
check("the ledger's own miss is a wrong plan",
      lambda: diagnose("no payment found with reference 'PMT-9999'") == "wrong_plan")
check("an unrecognised error is not guessed at",
      lambda: diagnose("kernel panic in the mainframe") == "unknown",
      "returning 'transient' by default is how a permanent failure gets retried forever")
check("a transient failure retries while attempts remain",
      lambda: respond_to("transient", attempts=1) == "retry")
check("retries are bounded -- the rung ends in escalation",
      lambda: respond_to("transient", attempts=3) == "escalate",
      "unbounded retry turns a blip into an outage on your side")
check("a wrong plan re-plans rather than retrying",
      lambda: respond_to("wrong_plan", attempts=0) == "replan")
check("an unknown error escalates", lambda: respond_to("unknown", attempts=0) == "escalate")

## Section 3 &mdash; Two agents meet the same failures

A retry-only agent and a diagnosing agent, run against a scripted sequence of failures. The
difference is not subtle.

In [ ]:
FAILURES = [
    "HTTP 503 service unavailable",                  # transient -- retry works
    "HTTP 503 service unavailable",
    "ok: {'ref': 'PMT-1003', 'status': 'held'}",     # succeeds on the third attempt
    "no payment found with reference 'PMT-0000'",    # permanent -- retry can never work
]

def run_retry_only(events, max_attempts=3):
    """Retries everything. The naive agent most teams ship first."""
    calls, attempts = 0, 0
    for e in events:
        calls += 1
        if e.startswith("ok:"):
            return {"calls": calls, "outcome": "success"}
        attempts += 1
        if attempts >= max_attempts:
            return {"calls": calls, "outcome": "gave up after retrying"}
    return {"calls": calls, "outcome": "exhausted events"}

def run_diagnosing(events, max_attempts=3):
    """Classifies each failure and takes the matching rung."""
    calls, attempts = 0, 0
    for e in events:
        calls += 1
        if e.startswith("ok:"):
            return {"calls": calls, "outcome": "success"}
        action = respond_to(diagnose(e), attempts)
        if action == "retry":
            attempts += 1
            continue
        return {"calls": calls, "outcome": action}
    return {"calls": calls, "outcome": "exhausted events"}

In [ ]:
# --- Self-check: Section 3
_transient_run = FAILURES[:3]
_permanent_first = ["no payment found with reference 'PMT-0000'"] + FAILURES[:3]

check("both agents ride out a transient blip",
      lambda: run_retry_only(_transient_run)["outcome"] == "success"
              and run_diagnosing(_transient_run)["outcome"] == "success")
check("the retry-only agent burns three calls on a permanent failure",
      lambda: run_retry_only(_permanent_first)["calls"] == 3)
check("the diagnosing agent re-plans on the first permanent failure",
      lambda: run_diagnosing(_permanent_first)["outcome"] == "replan")
check("...and it spends exactly one call to find that out",
      lambda: run_diagnosing(_permanent_first)["calls"] == 1,
      "the error said 'not found' on attempt one; nothing is learned by asking again")

for name, fn in (("retry-only", run_retry_only), ("diagnosing", run_diagnosing)):
    try:
        r = fn(_permanent_first)
        print(f"{name:12} calls={r['calls']}  outcome={r['outcome']}")
    except NameError:
        print("(fill in the blanks above, then re-run)"); break

## Section 4 &mdash; Plan, then validate it

Put the two halves together: decompose a goal, reject the sub-goals that cannot finish, and order
what survives with Lab 1.2's dependency rule.

In [ ]:
def validate_plan(subgoals: dict[str, list[str]]) -> dict:
    """Split a plan into the sub-goals worth running and the ones that cannot finish.

    Returns {"runnable": [...ordered...], "rejected": [...]}.
    """
    rejected = [s for s in subgoals if not is_finishable(s)]
    keep = {s: [d for d in deps if d not in rejected]
            for s, deps in subgoals.items() if s not in rejected}

    ordered, done = [], set()
    while len(ordered) < len(keep):
        progressed = False
        for name, deps in keep.items():
            if name in done:
                continue
            if all(d in done for d in deps):
                ordered.append(name); done.add(name); progressed = True
        if not progressed:
            raise ValueError("cycle in plan")
    return {"runnable": ordered, "rejected": rejected}

In [ ]:
# --- Self-check: Section 4
PLAN = {
    "Return the ledger record for the payment": [],
    "Understand the payment problem": [],                     # cannot finish
    "Return the policy text for its reason code": ["Return the ledger record for the payment"],
    "State whether policy reserves this for a human": ["Return the policy text for its reason code"],
    "Make sure nothing has been missed": [],                  # cannot finish
}

def _v():
    return validate_plan(PLAN)

check("both unfinishable sub-goals are rejected", lambda: len(_v()["rejected"]) == 2)
check("the three runnable sub-goals survive", lambda: len(_v()["runnable"]) == 3)
check("the ledger read comes first",
      lambda: _v()["runnable"][0].startswith("Return the ledger"))
check("dependencies are respected",
      lambda: (lambda o: o.index("Return the policy text for its reason code")
                       < o.index("State whether policy reserves this for a human"))(_v()["runnable"]))

try:
    v = _v()
    print("runnable:"); [print("   ", s) for s in v["runnable"]]
    print("rejected:"); [print("   ", s) for s in v["rejected"]]
except NameError:
    print("(fill in is_finishable above, then re-run)")

## Run it for real

Have the model decompose a goal, then put its plan through your validator. Models produce vague
sub-goals readily &mdash; this is the check that catches them before they become loops.

In [ ]:
if llm_ready():
    try:
        raw = ask(
            "Break this goal into 4 to 6 sub-goals, one per line, no numbering or bullets: "
            "'Determine who must action the exception on payment PMT-1005 and why.' "
            "Each sub-goal must start with one of: Return, Fetch, Compute, Classify, State, List."
        )
        proposed = [l.strip("-* \t") for l in raw.splitlines() if l.strip()]
        print("the model proposed:")
        for s in proposed:
            mark = "keep  " if is_finishable(s) else "REJECT"
            print(f"  [{mark}] {s}")
        bad = [s for s in proposed if not is_finishable(s)]
        print(f"\n{len(proposed) - len(bad)} of {len(proposed)} sub-goals are finishable.")
        if bad:
            print("Rejected because completion is not detectable from any output:")
            for s in bad:
                print("   " + s)
    except NameError:
        print("(fill in the blanks above, then re-run this cell)")

### Read it

Even told exactly which verbs to use, models slip in a "thoroughly" or an "as needed". Each one is
a step whose completion nothing can detect &mdash; and a step the agent will keep working on.

This validator is cheap and runs before any tokens are spent on execution. It is the highest
return-per-line check in the whole module.

In [ ]:
score()

## Your turn

1. `diagnose()` returns `unknown` for anything unrecognised, and `unknown` escalates. Argue for the
   opposite default, then say what would have to be true about your tools for it to be safe.
2. Rung 2 of the ladder &mdash; *try a different tool for the same sub-goal* &mdash; is not implemented.
   Add it, and decide where it sits between retry and re-plan.